# ROBUST04 Run 3 - Optimized (Pre-SPLADE)

## Optimizations Applied:
1. **k=10** (validated: MAP 0.2769)
2. **rerank_depth=1000** (optimal from tuning)
3. **Better RM3**: fb_docs=20, fb_terms=15, original_query_weight=0.6
4. **Consistent parameters** for train/test

## Current Performance:
- Pure RRF (BM25 + RM3 + MonoT5)
- Expected MAP: ~0.278-0.280

## Next Step:
After validating these improvements → Add SPLADE as 4th signal

In [ ]:
hugging_face_1bLLamaInstruct = "[REDACTED_HF_TOKEN]"

In [ ]:
from huggingface_hub import login

login(hugging_face_1bLLamaInstruct)

In [ ]:
# ============================================================================
# JAVA 21 SETUP
# ============================================================================

import os
import subprocess

print("Installing Java 21...")
!sudo apt-get update -qq
!sudo apt-get install -y openjdk-21-jdk-headless -qq

os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-21-openjdk-amd64"
os.environ["PATH"] = f"{os.environ['JAVA_HOME']}/bin:" + os.environ["PATH"]

!java -version
print("✓ Java 21 ready")

In [ ]:
# ============================================================================
# INSTALL PACKAGES
# ============================================================================

print("Installing packages...")
!pip install -q transformers>=4.46.0 pyserini==0.22.1 ir_measures torch sentence-transformers numpy
print("✓ All packages installed!")

In [ ]:
import os
import torch
import numpy as np
from collections import defaultdict
from pyserini.search.lucene import LuceneSearcher
from pyserini.index.lucene import IndexReader
import ir_measures
from ir_measures import MAP, nDCG, P
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
import warnings
warnings.filterwarnings('ignore')

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

In [ ]:
# ============================================================================
# GOOGLE DRIVE SETUP
# ============================================================================
from google.colab import drive

print("Mounting Google Drive...")
drive.mount('/content/drive')

BASE_DIR = '/content/drive/MyDrive/TEXT/FinalProject'
if os.path.exists(BASE_DIR):
    os.chdir(BASE_DIR)
    print(f"✓ Changed to: {BASE_DIR}")
else:
    print("⚠️  Directory not found, searching...")
    for path in ['/content/drive/MyDrive/TEXT/FinalProject', '/content/drive/MyDrive/FinalProject', '/content/drive/MyDrive']:
        if os.path.exists(os.path.join(path, 'queriesROBUST.txt')):
            os.chdir(path)
            print(f"✓ Found at: {path}")
            break

print(f"\nWorking directory: {os.getcwd()}")
print(f"✓ queriesROBUST.txt: {os.path.exists('queriesROBUST.txt')}")
print(f"✓ qrels_50_Queries: {os.path.exists('qrels_50_Queries')}")

## Configuration

In [ ]:
# OPTIMIZED CONFIGURATION (from parameter tuning)
USE_MONOT5 = True
OPTIMAL_K = 10        # Best from tuning (MAP 0.2769)
OPTIMAL_DEPTH = 1000  # Best from tuning

print(f"\n🎯 Optimized Configuration:")
print(f"  • MonoT5 Reranker: {USE_MONOT5}")
print(f"  • k_rrf: {OPTIMAL_K} (tuned)")
print(f"  • rerank_depth: {OPTIMAL_DEPTH} (tuned)")
print(f"  • RRF: Pure/Unweighted")
print(f"\n📈 Expected: MAP ~0.278-0.280")

## Load Index and Searchers

In [ ]:
print("Loading ROBUST04 index...")
index_reader = IndexReader.from_prebuilt_index('robust04')

# BM25 searcher
bm25_searcher = LuceneSearcher.from_prebuilt_index('robust04')
bm25_searcher.set_bm25(k1=0.9, b=0.4)

# RM3 searcher - OPTIMIZED PARAMETERS
rm3_searcher = LuceneSearcher.from_prebuilt_index('robust04')
rm3_searcher.set_bm25(k1=0.9, b=0.4)
rm3_searcher.set_rm3(fb_docs=20, fb_terms=15, original_query_weight=0.6)  # ✅ IMPROVED

print("✓ Configured BM25 + RM3")
print("  BM25: k1=0.9, b=0.4")
print("  RM3: fb_docs=20, fb_terms=15, original_query_weight=0.6 (OPTIMIZED)")

## Load Queries

In [ ]:
def load_queries(query_file):
    queries = {}
    with open(query_file, 'r', encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            parts = line.split(None, 1)
            if len(parts) == 2:
                queries[parts[0]] = parts[1]
    return queries

def load_qrels(qrel_file):
    qrels = defaultdict(dict)
    with open(qrel_file, 'r', encoding='utf-8') as f:
        for line in f:
            parts = line.strip().split()
            if len(parts) >= 4:
                qrels[parts[0]][parts[2]] = int(parts[3])
    return qrels

queries = load_queries('queriesROBUST.txt')
qrels = load_qrels('qrels_50_Queries')
train_qids = sorted(qrels.keys())
test_qids = sorted([q for q in queries if q not in qrels])

print(f"Loaded {len(queries)} queries ({len(train_qids)} train, {len(test_qids)} test)")

## Helper Functions

In [ ]:
def classify_query(query_text):
    """Classify query by length"""
    wc = len(query_text.split())
    return 'short' if wc <= 3 else 'medium' if wc <= 6 else 'long'

## Multi-Method Retrieval

In [ ]:
def multi_method_retrieval(query_text, k=1000):
    """Get results from BM25 and RM3"""
    bm25_hits = bm25_searcher.search(query_text, k=k)
    rm3_hits = rm3_searcher.search(query_text, k=k)

    bm25_results = [(h.docid, h.score) for h in bm25_hits]
    rm3_results = [(h.docid, h.score) for h in rm3_hits]

    return bm25_results, rm3_results

## Load MonoT5 Reranker

In [ ]:
monot5_model, monot5_tokenizer = None, None
if USE_MONOT5:
    print("Loading MonoT5 Reranker...")
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    try:
        mn = "castorini/monot5-base-msmarco-10k"
        monot5_tokenizer = AutoTokenizer.from_pretrained(mn)
        monot5_model = AutoModelForSeq2SeqLM.from_pretrained(mn).to(device)
        monot5_model.eval()
        print(f"  ✓ MonoT5 loaded successfully")
    except Exception as e:
        print(f"  ⚠️  MonoT5 load failed: {e}")
        USE_MONOT5 = False

## MonoT5 Reranker Function

In [ ]:
def rerank_monot5(query, doc_ids, batch_size=8):
    """MonoT5 reranking - returns dict of {doc_id: relevance_probability}"""
    if not USE_MONOT5 or monot5_model is None:
        return None

    doc_texts, valid_ids = [], []
    for did in doc_ids:
        try:
            doc = index_reader.doc(did)
            if doc and doc.raw():
                doc_texts.append(doc.raw()[:2000])
                valid_ids.append(did)
        except:
            pass

    if not doc_texts:
        return None

    prompts = [f"Query: {query} Document: {d} Relevant:" for d in doc_texts]
    true_token_id = monot5_tokenizer.encode("true")[0]
    false_token_id = monot5_tokenizer.encode("false")[0]

    all_scores = []
    for i in range(0, len(prompts), batch_size):
        batch = prompts[i:i+batch_size]
        try:
            inputs = monot5_tokenizer(batch, return_tensors="pt", padding=True, truncation=True, max_length=512).to(device)
            with torch.no_grad():
                outputs = monot5_model.generate(**inputs, max_new_tokens=1, return_dict_in_generate=True, output_scores=True)
                logits = outputs.scores[0]
                true_logits = logits[:, true_token_id]
                false_logits = logits[:, false_token_id]
                probs = torch.softmax(torch.stack([false_logits, true_logits], dim=1), dim=1)
                all_scores.extend(probs[:, 1].cpu().tolist())
        except Exception as e:
            all_scores.extend([0.5] * len(batch))

    return dict(zip(valid_ids, all_scores))

## Optimized Pipeline (Pure RRF with k=10)

In [ ]:
def optimized_pipeline(query, rerank_depth=OPTIMAL_DEPTH, k_rrf=OPTIMAL_K):
    """
    OPTIMIZED Pure RRF Pipeline
    
    Improvements:
    - k=10 (validated: MAP 0.2769)
    - rerank_depth=1000 (optimal)
    - Better RM3 parameters
    
    Pure RRF: BM25 + RM3 + MonoT5 (all weights = 1.0)
    """
    # Stage 1: Get BM25 and RM3 rankings
    bm25_results, rm3_results = multi_method_retrieval(query)

    bm25_ranking = {docid: rank for rank, (docid, _) in enumerate(bm25_results, 1)}
    rm3_ranking = {docid: rank for rank, (docid, _) in enumerate(rm3_results, 1)}

    # Stage 2: Get MonoT5 ranking for top docs
    monot5_ranking = {}
    if USE_MONOT5 and monot5_model is not None:
        top_k_docids = [docid for docid, _ in bm25_results[:rerank_depth]]
        monot5_scores = rerank_monot5(query, top_k_docids)

        if monot5_scores and len(monot5_scores) > 0:
            sorted_by_monot5 = sorted(monot5_scores.items(), key=lambda x: x[1], reverse=True)
            monot5_ranking = {docid: rank for rank, (docid, _) in enumerate(sorted_by_monot5, 1)}

    # Stage 3: Pure RRF Fusion (all weights = 1.0)
    all_docids = set(bm25_ranking.keys()) | set(rm3_ranking.keys())
    rrf_scores = {}

    for docid in all_docids:
        rrf_score = 0.0

        if docid in bm25_ranking:
            rrf_score += 1.0 / (k_rrf + bm25_ranking[docid])

        if docid in rm3_ranking:
            rrf_score += 1.0 / (k_rrf + rm3_ranking[docid])

        if docid in monot5_ranking:
            rrf_score += 1.0 / (k_rrf + monot5_ranking[docid])

        rrf_scores[docid] = rrf_score

    # Sort by RRF score
    final_ranking = sorted(rrf_scores.items(), key=lambda x: x[1], reverse=True)

    # Build results
    results = []
    for rank, (docid, score) in enumerate(final_ranking[:1000], 1):
        results.append((str(docid), float(score), int(rank)))

    return results

## Evaluate on Training Set

In [ ]:
print("="*70)
print("📊 EVALUATING OPTIMIZED PIPELINE ON TRAINING SET")
print("="*70)
print("Optimizations:")
print(f"  • k_rrf = {OPTIMAL_K} (tuned)")
print(f"  • rerank_depth = {OPTIMAL_DEPTH} (tuned)")
print(f"  • Better RM3: fb_docs=20, fb_terms=15, original_query_weight=0.6")
print("="*70)
print()

import time

train_results = []
start_time = time.time()

for i, qid in enumerate(train_qids, 1):
    query_text = queries[qid]
    query_type = classify_query(query_text)

    print(f"[{i}/{len(train_qids)}] Query {qid} ({query_type}): {query_text[:50]}...")

    try:
        results = optimized_pipeline(query_text)

        for docid, score, rank in results:
            train_results.append(ir_measures.ScoredDoc(
                query_id=str(qid),
                doc_id=str(docid),
                score=float(score)
            ))

        print(f"  ✓ Retrieved {len(results)} docs")
    except Exception as e:
        print(f"  ✗ Error: {str(e)[:100]}")
        continue

total_time = time.time() - start_time

# Convert qrels
qrels_list = []
for qid, doc_rels in qrels.items():
    for docid, rel in doc_rels.items():
        qrels_list.append(ir_measures.Qrel(
            query_id=str(qid),
            doc_id=str(docid),
            relevance=int(rel)
        ))

# Calculate metrics
print("\n" + "="*70)
print("📈 COMPUTING METRICS...")
print("="*70)

metrics = [MAP, nDCG@10, nDCG@20, P@10, P@20]
results_metrics = ir_measures.calc_aggregate(metrics, qrels_list, train_results)

print("\n" + "="*70)
print("🏆 OPTIMIZED PIPELINE PERFORMANCE")
print("="*70)
print("\n📊 Your Scores:")
print(f"  MAP:      {results_metrics[MAP]:.4f}  ← Main metric")
print(f"  nDCG@10:  {results_metrics[nDCG@10]:.4f}")
print(f"  nDCG@20:  {results_metrics[nDCG@20]:.4f}")
print(f"  P@10:     {results_metrics[P@10]:.4f}")
print(f"  P@20:     {results_metrics[P@20]:.4f}")

# Comparison
original_map = 0.2738
tuned_k10_map = 0.2769
current_map = results_metrics[MAP]

print(f"\n📈 Comparison:")
print(f"  Original (k=30, depth=300, old RM3): {original_map:.4f}")
print(f"  Tuned k=10 (old RM3): {tuned_k10_map:.4f}")
print(f"  Optimized (k=10, depth=1000, new RM3): {current_map:.4f}")

if current_map >= tuned_k10_map:
    improvement = current_map - tuned_k10_map
    print(f"\n  ✅ Improvement: +{improvement:.4f} from RM3 optimization")
else:
    decline = tuned_k10_map - current_map
    print(f"\n  ⚠️  Decline: -{decline:.4f} (RM3 may need adjustment)")

print(f"\n⏱️  Time: {total_time:.1f}s ({total_time/len(train_qids):.1f}s per query)")

print("\n" + "="*70)
print("🎯 NEXT STEP: ADD SPLADE")
print("="*70)
print(f"Current baseline: MAP {current_map:.4f}")
print(f"Expected with SPLADE: MAP ~{current_map + 0.06:.2f} (target: 0.33-0.35)")
print("="*70)

## Generate Test Results (Optimized)

In [ ]:
import time

print("="*70)
print("🚀 GENERATING TEST RESULTS - OPTIMIZED PIPELINE")
print("="*70)
print(f"Configuration:")
print(f"  • Pure RRF (BM25 + RM3 + MonoT5)")
print(f"  • k = {OPTIMAL_K}")
print(f"  • rerank_depth = {OPTIMAL_DEPTH}")
print(f"  • RM3: fb_docs=20, fb_terms=15, original_query_weight=0.6")
print(f"  • Test queries: {len(test_qids)}")
print("="*70)
print()

test_results = []
start_time = time.time()

for i, qid in enumerate(test_qids, 1):
    query_text = queries[qid]
    query_type = classify_query(query_text)

    print(f"[{i}/{len(test_qids)}] Query {qid} ({query_type}): {query_text[:60]}...")

    query_start = time.time()
    results = optimized_pipeline(query_text)  # Uses OPTIMAL_K and OPTIMAL_DEPTH by default
    query_time = time.time() - query_start

    for docid, score, rank in results:
        test_results.append({
            'qid': str(qid),
            'docid': str(docid),
            'rank': int(rank),
            'score': float(score)
        })

    print(f"  ✓ Retrieved {len(results)} docs in {query_time:.1f}s")

    if i % 10 == 0:
        elapsed = time.time() - start_time
        avg_time = elapsed / i
        remaining = (len(test_qids) - i) * avg_time
        print(f"\n  Progress: {i}/{len(test_qids)} ({i/len(test_qids)*100:.1f}%)")
        print(f"  Elapsed: {elapsed/60:.1f}min, Remaining: {remaining/60:.1f}min\n")

total_time = time.time() - start_time

print("\n" + "="*70)
print("✓ GENERATION COMPLETE!")
print("="*70)
print(f"  Queries: {len(test_qids)}")
print(f"  Rankings: {len(test_results):,}")
print(f"  Time: {total_time/60:.1f} minutes")
print("="*70)

## Save Results

In [ ]:
with open('run_3_optimized.res', 'w') as f:
    for r in test_results:
        f.write(f"{r['qid']} Q0 {r['docid']} {r['rank']} {r['score']:.6f} run3_optimized\n")

print("✓ Saved to run_3_optimized.res")
print("\nFirst 5 lines:")
with open('run_3_optimized.res', 'r') as f:
    for i, line in enumerate(f):
        if i < 5:
            print(line.strip())
        else:
            break

print("\n" + "="*70)
print("📝 PHASE 1 COMPLETE - OPTIMIZATIONS APPLIED")
print("="*70)
print("Improvements:")
print("  ✅ k=10 (validated)")
print("  ✅ rerank_depth=1000 (optimal)")
print("  ✅ Better RM3 parameters")
print("  ✅ Consistent train/test configuration")
print("\nNext: Add SPLADE as 4th signal for MAP 0.33-0.35")
print("="*70)